# QLoRA Fine-tuning

`1. VLM 학습 데이터 준비.ipynb`가 HuggingFace Hub에 업로드한 ChartQA_KOR 데이터셋을 내려받아
**Qwen3-VL-2B-Instruct**를 QLoRA(4bit 양자화 + LoRA)로 fine-tuning합니다.

> ⚠️ RunPod GPU 인스턴스(24GB VRAM급 이상 권장)에서 실행하세요. CPU 환경에서는 4bit 양자화 로딩이 정상 동작하지 않습니다.

## 사전 준비
1. `.env.example`을 `.env`로 복사하고 `HF_TOKEN`, `HF_DATASET_REPO_ID`(노트북 1에서 업로드한 repo)를 채워넣기
```bash
cp .env.example .env
uv sync
huggingface-cli login   # Qwen3-VL 다운로드용
```


In [1]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from dotenv import load_dotenv
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

import vlm_finetune_utils as vu

load_dotenv()
HF_TOKEN = os.environ["HF_TOKEN"]
HF_DATASET_REPO_ID = os.environ["HF_DATASET_REPO_ID"]

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

# ChartQA_KOR 원본 이미지는 최대 3000px에 달해, Qwen3-VL의 동적 해상도 비전 인코더가
# 매우 많은 이미지 토큰을 만들어 GPU 메모리를 크게 씁니다. min/max_pixels로 상한을 둡니다.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28


## 1. 4bit 양자화로 base 모델 로드
QLoRA 방식대로, 원본 가중치는 4bit로 양자화해서 올리고 LoRA(A, B) 파라미터만 원래 정밀도(bf16)로 학습합니다.
`prepare_model_for_kbit_training`은 양자화된 모델에 gradient checkpointing을 안전하게 걸어주고
(레이어 출력 재계산으로 활성화 메모리를 절약), LayerNorm 등을 fp32로 캐스팅해 학습을 안정화합니다.

In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

## 2. LoRA 설정
언어 모델의 attention/MLP 프로젝션 레이어에 저랭크 어댑터(A, B)를 붙입니다. `r`(랭크)이 클수록 표현력은 늘지만 학습 파라미터도 늘어납니다.

In [3]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable, total = vu.count_trainable_params(model)
print(f"학습 파라미터: {trainable:,} / 전체 {total:,} ({trainable / total:.4%})")


학습 파라미터: 6,422,528 / 전체 1,227,984,896 (0.5230%)


## 3. 데이터셋 다운로드 및 전처리
`HF_DATASET_REPO_ID`에서 데이터셋을 내려받아 `data/`에 저장한 뒤, `to_chat_messages`로 각
레코드를 processor의 chat template 입력 형태로 바꿉니다.
전체 1,600장을 다 쓰면 강의 시간 내에 끝나지 않을 수 있어, 실습에서는 `N_TRAIN`개만 서브샘플링합니다
(실제 프로젝트라면 전체 데이터를 사용하세요).

In [4]:
N_TRAIN = 200  # 강의 시간 내 실습을 위한 서브셋 크기 (최대 1,600)

vu.download_dataset_from_hub("data", HF_DATASET_REPO_ID, token=HF_TOKEN)

train_records = vu.load_jsonl("data/train/annotations.jsonl")[:N_TRAIN]
val_records = vu.load_jsonl("data/val/annotations.jsonl")

train_messages = [vu.to_chat_messages(r, "data/train") for r in train_records]
train_messages[0]


'data'에 이미 다운로드되어 있어 건너뜁니다.


[{'role': 'user',
  'content': [{'type': 'image', 'image': 'data/train/images/0000.png'},
   {'type': 'text',
    'text': '그래프에서 주황색은 무엇을 나타냅니까?\n질문에 대한 답만 아주 간결하게 말하세요. 숫자면 숫자만, 단답이면 단답만 답하고 설명은 덧붙이지 마세요.'}]},
 {'role': 'assistant', 'content': [{'type': 'text', 'text': '탈레반'}]}]

In [5]:
from qwen_vl_utils import process_vision_info


def collate_fn(batch_messages):
    texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch_messages
    ]
    image_inputs, _ = process_vision_info(
        [msg for messages in batch_messages for msg in messages]
    )
    batch = processor(
        text=texts, images=image_inputs, return_tensors="pt", padding=True
    )
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch


## 4. 학습
서브셋(`N_TRAIN`장)이라 몇 epoch만으로도 패턴이 학습되는지 확인할 수 있습니다.

In [6]:
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

model.config.use_cache = False  # gradient checkpointing과 함께 쓸 때는 꺼야 함

loader = DataLoader(train_messages, batch_size=2, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-4
)
num_epochs = 10
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=5, num_training_steps=num_epochs * len(loader)
)


In [7]:
model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in loader:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"epoch {epoch + 1}/{num_epochs} - loss {total_loss / len(loader):.4f}")


/app/.venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/app/.venv/lib/python3.11/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


epoch 1/10 - loss 9.0981
epoch 2/10 - loss 6.5995
epoch 3/10 - loss 6.5384
epoch 4/10 - loss 6.5175
epoch 5/10 - loss 6.5057
epoch 6/10 - loss 6.4967
epoch 7/10 - loss 6.4923
epoch 8/10 - loss 6.4878
epoch 9/10 - loss 6.4861
epoch 10/10 - loss 6.4847


## 5. 어댑터 저장
학습된 LoRA 가중치(A, B)만 저장합니다 — 원본 모델은 그대로 두므로 결과물이 작습니다.

In [8]:
ADAPTER_DIR = "outputs/qlora-chartqa-adapter"
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print("저장 완료:", ADAPTER_DIR)


저장 완료: outputs/qlora-chartqa-adapter


## LoRA와의 차이 정리
- 이 노트북은 `BitsAndBytesConfig(load_in_4bit=True)`로 **QLoRA**를 사용했습니다.
- 순수 LoRA로 하려면 `quantization_config` 없이 `torch_dtype=torch.bfloat16`으로만 모델을 로드하면 됩니다 — VRAM이 더 필요한 대신 양자화 오차가 없어 약간 더 안정적입니다.
- 다음 노트북에서 이 어댑터를 base 모델과 비교 평가합니다.